# 08 — Hardware Supply Chain Security

## What This Is
Supply chain attacks compromise hardware or firmware before it reaches the end user. Hardware trojans insert malicious logic into chips during manufacture. Firmware implants persist through OS reinstalls. SolarWinds and ASUS Live Update showed that software supply chains are equally vulnerable.

## Why It Matters
NSA ANT catalog (2013 Snowden leak) documented hardware implants including COTTONMOUTH (USB implant), IRONCHEF (motherboard firmware persistence), and FEEDTROUGH (Juniper firewall implant). Nation-state actors routinely target hardware supply chains for intelligence collection.

## Where the Community Stands
SBOM (Software Bill of Materials) and HBoM (Hardware Bill of Materials) are regulatory requirements (US Executive Order 14028). NIST SP 800-161 provides supply chain risk management guidance. Transparency logs (Sigstore) provide software artifact attestation. Hardware inspection (X-ray, die photography) detects trojan insertions.

## Authorised Context
All trojan detection and integrity verification discussed here is defensive — verifying the integrity of hardware and software you own or are responsible for protecting.

In [ ]:
import hashlib
import json
import secrets
import hmac
from dataclasses import dataclass, field
from typing import List, Dict, Optional

# --- Software Bill of Materials (SBOM) ---

@dataclass
class SBOMEntry:
    name:        str
    version:     str
    supplier:    str
    hash_sha256: str
    license:     str
    cve_known:   List[str] = field(default_factory=list)
    is_oss:      bool = True

class SBOM:
    def __init__(self, product: str, version: str):
        self.product   = product
        self.version   = version
        self.components: List[SBOMEntry] = []
        self._signature: Optional[bytes] = None

    def add(self, entry: SBOMEntry) -> None:
        self.components.append(entry)

    def to_json(self) -> str:
        return json.dumps({
            'product': self.product, 'version': self.version,
            'components': [
                {'name': c.name, 'version': c.version, 'supplier': c.supplier,
                 'sha256': c.hash_sha256, 'license': c.license,
                 'cves': c.cve_known}
                for c in self.components
            ]
        }, indent=2)

    def sign(self, key: bytes) -> None:
        self._signature = hmac.new(key, self.to_json().encode(), hashlib.sha256).digest()

    def verify(self, key: bytes) -> bool:
        if not self._signature:
            return False
        expected = hmac.new(key, self.to_json().encode(), hashlib.sha256).digest()
        return hmac.compare_digest(self._signature, expected)

# Build an SBOM for a hypothetical embedded firmware
sbom = SBOM('EmbeddedRouter-FW', '3.2.1')
sbom.add(SBOMEntry('OpenSSL',    '3.0.8', 'OpenSSL Foundation',
                   hashlib.sha256(b'openssl-3.0.8.tar.gz').hexdigest(),
                   'Apache-2.0', cve_known=['CVE-2023-0286']))
sbom.add(SBOMEntry('BusyBox',    '1.36.1','BusyBox Project',
                   hashlib.sha256(b'busybox-1.36.1.tar.bz2').hexdigest(),
                   'GPL-2.0'))
sbom.add(SBOMEntry('uClibc-ng',  '1.0.44','uClibc-ng Project',
                   hashlib.sha256(b'uClibc-ng-1.0.44.tar.xz').hexdigest(),
                   'LGPL-2.1'))
sbom.add(SBOMEntry('dnsmasq',    '2.89',  'Simon Kelley',
                   hashlib.sha256(b'dnsmasq-2.89.tar.gz').hexdigest(),
                   'GPL-2.0', cve_known=['CVE-2023-28450']))

KEY = secrets.token_bytes(32)
sbom.sign(KEY)

print(f'SBOM for {sbom.product} v{sbom.version}: {len(sbom.components)} components')
print(f'Signature valid: {sbom.verify(KEY)}')
print('\nComponents with known CVEs:')
for c in sbom.components:
    if c.cve_known:
        print(f'  {c.name} {c.version}: {c.cve_known}')

In [ ]:
# Firmware integrity verification
def compute_firmware_hash(firmware_blob: bytes, region_map: Dict[str, tuple]) -> Dict[str, str]:
    """Hash individual firmware regions for granular integrity checking."""
    region_hashes = {}
    for region_name, (start, end) in region_map.items():
        region_data = firmware_blob[start:end]
        region_hashes[region_name] = hashlib.sha256(region_data).hexdigest()
    region_hashes['full_firmware'] = hashlib.sha256(firmware_blob).hexdigest()
    return region_hashes

# Simulate firmware regions
FW_SIZE    = 16 * 1024 * 1024  # 16MB
rng        = __import__('random').Random(42)
firmware   = bytes(rng.randint(0,255) for _ in range(FW_SIZE))

REGION_MAP = {
    'bootloader':    (0,          512*1024),
    'kernel':        (512*1024,   4*1024*1024),
    'rootfs':        (4*1024*1024,12*1024*1024),
    'config':        (12*1024*1024,14*1024*1024),
    'nvram':         (14*1024*1024,16*1024*1024),
}

golden_hashes = compute_firmware_hash(firmware, REGION_MAP)
print('Firmware region hashes (golden):')
for region, h in golden_hashes.items():
    print(f'  {region:<15} {h[:32]}...')

# Simulate tampered firmware (bootloader modified)
tampered = bytearray(firmware)
tampered[1000] ^= 0xFF  # flip a byte in bootloader region
tampered = bytes(tampered)

tampered_hashes = compute_firmware_hash(tampered, REGION_MAP)
print('\nIntegrity check (tampered firmware):')
for region in golden_hashes:
    match = golden_hashes[region] == tampered_hashes[region]
    status = 'OK' if match else 'TAMPERED!'
    print(f'  {region:<15} {status}')

## Hardware Trojan Detection
Hardware trojans are malicious modifications inserted during chip design or fabrication. Detection techniques include: functional testing (rarely finds trojans), side-channel analysis (power/timing anomalies), and logic testing with rare-trigger coverage.

In [ ]:
# Hardware Trojan conceptual model
@dataclass
class CircuitGate:
    gate_id: str
    gate_type: str    # AND/OR/NOT/XOR/NAND/NOR
    inputs: List[str]
    is_trojan: bool = False
    trigger_condition: Optional[str] = None

def simulate_trojan_detection(circuit: List[CircuitGate],
                               n_tests: int = 10000,
                               rng = None) -> Dict:
    """Estimate probability of activating trojan trigger via random testing."""
    if rng is None:
        rng = __import__('random').Random(0)

    trojan_gates   = [g for g in circuit if g.is_trojan]
    normal_gates   = [g for g in circuit if not g.is_trojan]
    n_inputs       = 16  # simulated primary inputs

    if not trojan_gates:
        return {'trojans': 0, 'detected': 0, 'detection_rate': 0.0}

    detections = 0
    for _ in range(n_tests):
        # Random input vector
        inputs = [rng.randint(0,1) for _ in range(n_inputs)]
        # Each trojan has a rare trigger (e.g., all 16 inputs = 1)
        for g in trojan_gates:
            if g.trigger_condition == 'all_ones' and all(inputs):
                detections += 1
            elif g.trigger_condition == 'specific_pattern':
                # Pattern: inputs[0..3] = 1010
                if inputs[:4] == [1,0,1,0]:
                    detections += 1

    rate = detections / n_tests
    return {'trojans': len(trojan_gates), 'activations': detections,
            'tests': n_tests, 'detection_rate': round(rate, 6)}

# Simulate a circuit with embedded trojan
rng = __import__('random').Random(42)
circuit = [
    CircuitGate('G1', 'AND',  ['IN0','IN1']),
    CircuitGate('G2', 'OR',   ['IN2','IN3']),
    CircuitGate('G3', 'XOR',  ['G1', 'G2']),
    CircuitGate('G4', 'AND',  ['G3', 'IN4']),
    # Trojan: triggers when all 16 inputs are 1 (probability = 2^-16 ≈ 0.0015%)
    CircuitGate('T1', 'AND',  ['IN0','IN1','IN2','IN3'], is_trojan=True,
                trigger_condition='all_ones'),
]

results = simulate_trojan_detection(circuit, n_tests=100000, rng=rng)
prob_random = (0.5 ** 16)
print(f'Hardware Trojan Detection:')
print(f'  Trojan trigger: all 16 inputs = 1')
print(f'  Theoretical probability: {prob_random:.6f} ({prob_random*100:.4f}%)')
print(f'  Random test detection:   {results["detection_rate"]:.6f}')
print(f'  Tests to 99% confidence: ~{int(0.99/prob_random):,}')
print(f'\n  Conclusion: random testing alone cannot find rare-trigger trojans')
print(f'  Solution: formal verification + side-channel profiling + optical inspection')